# ARBITER — Guided fine-tuning

**Goal:** adapt ARBITER to your decisions, monitor training, and compare the adapted model against the base.

### How to use this notebook

1. Run cells in order with **Shift + Enter**.
2. Set paths and hyperparameters in **Step 2**.
3. Prepare your files and run the checks in **Step 3**.
4. Set **`RUN_WORKFLOW = True`** in Step 2 and run the remaining cells to execute the full workflow. You can override individual action flags to run only selected steps.

**By default, “Run All” does not download data, train a model, or launch benchmarks.** Action cells preview their commands.

| Step | Expected result |
|---|---|
| 1–2 · Setup and parameters | Environment detected and paths configured |
| 3 · Data | Readable distributions and a successful audit |
| 4 · Preflight | Tokenized inputs within the length limit |
| 5–6 · Training and monitoring | Resume checkpoint, best model, and TensorBoard logs |
| 7–8 · Comparison | Predictions with expected answers, then before/after benchmarks |

The notebook can **build 80,000 training + 8,000 validation decisions directly from pinned public training sources**. Data is downloaded/generated locally, not bundled in Git. Set `DATASET_MODE = "80k"` (default), `"typed"` for the smaller source, or `"custom"` for your own JSONL files. This downloadable recipe uses published procedural tasks in place of the originally proposed new human-annotated policies. [Full plan and sources](../docs/targeted_finetuning.md).


## 1. Check your environment

Open this notebook from the ARBITER repository or its `notebooks/` directory. Select the Python kernel for your virtual environment.

If dependencies are missing, run **in a terminal at the repository root**:

```bash
python -m pip install -e '.[finetuning,benchmark]'
python -m jupyterlab
```

The following cell does not load the model or use the GPU.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime
from importlib.util import find_spec
from html import escape
import gc
import json
import math
import shlex
import subprocess
import sys
from IPython.display import HTML, Markdown, display

ROOT = next((d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (d / "finetuning/train.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from the ARBITER repository or its notebooks/ directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def table(headers, rows):
    head = "".join(f"<th>{escape(str(h))}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(v))}</td>" for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def launch(command, enabled=False):
    print(shlex.join(command))
    if enabled:
        subprocess.run(command, cwd=ROOT, check=True)
    else:
        print("Preview only: enable this cell’s execution flag to run.")

print("Repository:", ROOT)
print("Python:", sys.executable)
missing = [name for name in ("torch", "transformers", "safetensors", "datasets", "tensorboard") if find_spec(name) is None]
print("Dependencies:", "OK" if not missing else "Install: " + ", ".join(missing))

Repository: .
Python: ./.venv/bin/python
Dependencies: OK


## 2. Set paths and hyperparameters

**This is the main configuration cell.** The default data paths match the 80k builder. Both preparation and training use `MAX_LENGTH`; inputs exceeding it are filtered during building, with counts reported. Set `RUN_WORKFLOW = True` to enable source preparation, preflight, training, comparison, and benchmarks in order. Paths are relative to the repository. Use a distinct run name for each experiment; the base model is never overwritten.

| Parameter | Default | Meaning |
|---|---:|---|
| `EPOCHS` | 1 | Passes through the training data |
| `BATCH_SIZE` | 4 | Decisions processed simultaneously |
| `ACCUMULATION` | 8 | Microbatches per update; effective batch size = 32 |
| `MAX_LENGTH` | 1024 | Total token budget: context + instruction + options |
| `BACKBONE_LR` / `HEAD_LR` | 5e-6 / 2e-5 | Learning rates for the encoder and decision head |
| `CLIP` | 3 | Global gradient norm limit |
| `EVAL_EVERY` | 250 | Optimizer steps between validation/checkpoint saves |

`DETERMINISTIC = True` enables the mode tested for exact resume on the same GPU/software environment; it can affect speed. `BENCHMARK_LIMIT = 0` runs full evaluations. A positive limit is only a technical smoke test.

On CUDA, this trainer uses BF16: choose compatible hardware. If memory runs out, reduce the microbatch and increase accumulation to preserve the effective batch size. Runtime and memory requirements still need to be measured.


In [ ]:
RUN_WORKFLOW = False  # Set True to run the complete workflow after reviewing this configuration.
DATASET_MODE = "80k"  # "80k", "typed", or "custom"
RUN_NAME = "arbiter-targeted-80k-v1"
assert DATASET_MODE in ("80k", "typed", "custom")
DATA_DIR = ROOT / "results/adaptation" / DATASET_MODE
BASE = ROOT / "checkpoints/arbiter-272m"
TRAIN = DATA_DIR / "train.jsonl"
VALIDATION = DATA_DIR / "validation.jsonl"
HELDOUT = [ROOT / "benchmark/data/cases.jsonl",
           ROOT / "benchmark/data/public_requested_v1/cases.jsonl"]
OUTPUT = ROOT / "results" / RUN_NAME

DEVICE = "cuda"
EPOCHS = 1
BATCH_SIZE = 4
ACCUMULATION = 8
MAX_LENGTH = 1024
BACKBONE_LR = 5e-6
HEAD_LR = 2e-5
CLIP = 3
EVAL_EVERY = 250
SEED = 13
DETERMINISTIC = True  # Required for exact same-environment resume checks.
BENCHMARK_LIMIT = 0  # 0 = full benchmark; >0 = smoke test cases per suite, not publishable scores.

assert all(v > 0 for v in (EPOCHS, BATCH_SIZE, ACCUMULATION, MAX_LENGTH, EVAL_EVERY, CLIP))
assert OUTPUT.resolve() != BASE.resolve(), "The output directory must differ from the base directory."
table(["Resource", "Path", "Available"],
      [(name, path.relative_to(ROOT) if path.is_relative_to(ROOT) else path, "Yes" if path.exists() else "No")
       for name, path in [("Base", BASE), ("Train", TRAIN), ("Validation", VALIDATION),
                          *[(f"Held-out test {i+1}", path) for i, path in enumerate(HELDOUT)]]])
print("Effective batch size:", BATCH_SIZE * ACCUMULATION)
print("Output:", OUTPUT)

## 3. Prepare and inspect your data

### Required format

A **JSONL** file contains one JSON object per line: **one question per line**. Questions and translations from the same scenario share a `group_id` and stay in the same split.

- `context`: the text or state to assess, serialized as a string.
- `instruction`: the specific question to answer.
- `question_type`: `choice`, `noul`, or `score`.
- `criteria`: answer identifiers and descriptions.
- `target`: target probability for **every** answer; probabilities sum to 1.
- `family`: group/subfamily, such as `policy/escalation` or `nli/fr`.
- Provenance: `source`, `source_revision`, `source_split`, `license`.

For `noul`, use `false` followed by `true`. For `score`, order criteria from lowest to highest; the output is an expectation over positions 0 to K−1.

### Build the dataset in this notebook

The next cell downloads pinned **training splits**, converts their native labels, selects unique requests, reserves validation groups, checks token lengths, and writes `train.jsonl`, `validation.jsonl`, and a manifest.

| Training group | Decisions | Source |
|---|---:|---|
| Typed/workflows | 16,000 | Official Typed train, completed with procedural state/evidence tasks |
| NLI | 24,000 | XNLI translated train, all 15 languages kept together per premise |
| Policy decisions | 24,000 | Published policy applicability, uncertainty and taxonomy routing; 12k choice + 12k noul |
| Routing replay | 16,000 | 8k Banking77 + 8k MASSIVE (14 supported languages) |
| **Total** | **80,000** | Plus **8,000** grouped validation decisions |

Set **`BUILD_DATASET = True`** in the next cell to construct only the data. Set **`RUN_WORKFLOW = True`** in Step 2 to build and continue through training and evaluation.

There is no paid API, LLM-generated labeling or duplication to meet quotas. If insufficient eligible data remains, the build fails explicitly. A completed dataset is reused only if its manifest/configuration/file hashes match. The builder runs on CPU and loads only the tokenizer/config, not model weights.

**What this recipe means:** procedural scenarios are published synthetic examples, not newly reviewed human cases. Replay here focuses on routing; retention on other tasks still needs evaluation. The NLI train translations vary in quality, and their source has unspecified license metadata: check upstream terms before redistribution. Existing exposure in the base is not ruled out.

Fast Decisions evaluation files are never used as training sources. Held-out contexts and cross-split exact matches are filtered from the new corpus; benchmark files remain unchanged. The audit does not detect all paraphrases or template overlap. See the [80k builder documentation](../docs/build_80k.md).


In [ ]:
BUILD_DATASET = RUN_WORKFLOW  # Set True here to build data only, while keeping RUN_WORKFLOW=False.
if DATASET_MODE == "80k":
    prepare_command = [sys.executable, "-m", "finetuning.build_80k", "--model", str(BASE),
                       "--heldout", *map(str, HELDOUT), "--output", str(DATA_DIR),
                       "--seed", str(SEED), "--max-length", str(MAX_LENGTH)]
    launch(prepare_command, BUILD_DATASET)
elif DATASET_MODE == "typed":
    prepare_command = [sys.executable, "-m", "finetuning.prepare_sources", "--source", "typed",
                       "--revision", "d0e2f0c42fef86cc15d1688d25a19f5ba7c85b18",
                       "--seed", str(SEED), "--output", str(DATA_DIR)]
    if (TRAIN.is_file() and VALIDATION.is_file()):
        print("Typed files already exist; the next cell audits them.")
    else:
        launch(prepare_command, BUILD_DATASET)
else:
    print("Custom mode: provide TRAIN and VALIDATION files using the documented schema.")

build_manifest = DATA_DIR / "manifest.json"
if build_manifest.is_file():
    build_info = json.loads(build_manifest.read_text())
    table(["Split", "Decisions", "Semantic groups"],
          [(split, sum(counts.values()), build_info["groups"][split])
           for split, counts in build_info["counts"].items()])
    table(["Source", "Training decisions"], sorted(build_info["source_counts"]["train"].items()))
    for limitation in build_info["limitations"]:
        print("Note:", limitation)


In [ ]:
AUDIT_OK = False
train_rows, val_rows = [], []
required = [TRAIN, VALIDATION, *HELDOUT]
missing_files = [path for path in required if not path.is_file()]
if missing_files:
    print("Data is not ready yet. Missing files:")
    for path in missing_files:
        print(" -", path)
else:
    from finetuning.data import read_rows, audit
    try:
        train_rows, val_rows = read_rows(TRAIN), read_rows(VALIDATION)
        audit_report = audit(train_rows, val_rows, HELDOUT)
        families = sorted({r["family"] for r in train_rows + val_rows})
        tc, vc = Counter(r["family"] for r in train_rows), Counter(r["family"] for r in val_rows)
        table(["Family", "Train", "Validation"], [(f, tc[f], vc[f]) for f in families])
        table(["Type", "Train", "Validation"],
              [(kind, sum(r["question_type"] == kind for r in train_rows),
                sum(r["question_type"] == kind for r in val_rows)) for kind in ("choice", "noul", "score")])
        if set(tc) != set(vc):
            raise ValueError("Each family must appear in both training and validation.")
        AUDIT_OK = True
        print("Automatic audit passed — this does not replace semantic review.")
        print(f"{len(train_rows):,} training decisions; {len(val_rows):,} validation.")
        steps = math.ceil(len(train_rows) / (BATCH_SIZE * ACCUMULATION)) * EPOCHS
        print(f"Budget: {steps:,} optimizer steps, plus validation and checkpoint saves.")
    except (ValueError, TypeError, KeyError) as error:
        print("Audit blocked:", error)
        print("Fix the data, then rerun this cell. Do not remove benchmark rows.")

### Inspect an example and its expected answer

Check that the instruction, criteria, and target probabilities describe the intended decision. The first maximum is displayed as the expected answer; all target probabilities remain visible.


In [ ]:
EXAMPLE_INDEX = 0
if train_rows:
    example = train_rows[EXAMPLE_INDEX]
    display(Markdown("**Context**"))
    print(example["context"])
    print("Question:", example["instruction"])
    print("Type:", example["question_type"], "| Group:", example["group_id"])
    expected = max(example["target"], key=example["target"].get)
    table(["Option", "Description", "Target", "Expected answer"],
          [(k, v, f'{example["target"][k]:.1%}', "✓" if k == expected else "")
           for k, v in example["criteria"].items()])
else:
    print("An example will appear once the files are available.")

## 4. Check tokenization before training

This step loads the base **on CPU**, audits the files, and tokenizes all decisions. No forward or backward pass. It may take time and RAM.

**Expected result:** the command finishes without errors. Inputs exceeding `MAX_LENGTH` are rejected: increase the budget if appropriate, or revise the text without removing decisive evidence. Each launch creates a new preflight output directory.


In [ ]:
RUN_PREFLIGHT = RUN_WORKFLOW
common = [sys.executable, "-m", "finetuning.train", "--model", str(BASE),
          "--train", str(TRAIN), "--validation", str(VALIDATION), "--heldout", *map(str, HELDOUT)]
settings = ["--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
            "--accumulation", str(ACCUMULATION), "--max-length", str(MAX_LENGTH),
            "--backbone-lr", str(BACKBONE_LR), "--head-lr", str(HEAD_LR),
            "--clip", str(CLIP), "--eval-every", str(EVAL_EVERY), "--seed", str(SEED)]
if DETERMINISTIC:
    settings += ["--deterministic"]
preflight_output = ROOT / "results/adaptation" / ("preflight-" + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
if RUN_PREFLIGHT and not AUDIT_OK:
    print("Run the data audit in Step 3 first.")
else:
    launch(common + settings + ["--output", str(preflight_output), "--validate-only"], RUN_PREFLIGHT)

## 5. Start or resume fine-tuning

With `RUN_WORKFLOW = True`, this cell starts training. Alternatively, override `RUN_TRAINING = True` here to run just this step. **The kernel stays busy during training.** You can also copy the displayed command into a terminal.

- New run: `RESUME = False` and an empty or nonexistent output directory.
- Resume: `RESUME = True`, the same directory, parameters, data, and code.
- To stop: use **Interrupt Kernel**. Resuming starts from the last completed checkpoint; unsaved steps are recomputed.

| Output | Purpose |
|---|---|
| `latest.pt` | Latest resume state: weights, optimizer, scheduler, and RNG |
| `best/` | Inference package selected on validation |
| `validation.jsonl` | Per-family validation history |
| `tensorboard/` | Training curves |
| `manifest.json` | Configuration and file hashes |

The best model may remain the step-zero base if adaptation does not improve it. Validation selects the checkpoint, not the public benchmark.


In [ ]:
RUN_TRAINING = RUN_WORKFLOW
RESUME = False
command = common + settings + ["--output", str(OUTPUT), "--device", DEVICE]
if RESUME:
    command += ["--resume"]
if RUN_TRAINING and not AUDIT_OK:
    print("Run the data audit in Step 3 first.")
else:
    launch(command, RUN_TRAINING)

## 6. Monitor training

Run the TensorBoard command below **in a second terminal**. On the local machine, open `http://localhost:6006`. From another machine, use the reachable server address.

| Curve | Interpretation |
|---|---|
| `Loss/CE` | Error against target probabilities; lower is better |
| `Loss/Ordinal_MAE` | Expected-score error; applies only to `score` |
| `Gradient/pre_clip_norm` | Global norm before clipping |
| `Gradient/clip_scale` | 1 = no reduction; below 1 = gradients scaled down |
| `Validation/<family>/accuracy` | Correct decisions per family |
| `Validation/macro_ce` | Selection metric, balanced across groups |

Training values logged every 25 steps describe that accumulation window, not an average of the previous 25 windows. Oscillation alone does not establish a problem: focus on validation and skill retention.


In [ ]:
print(shlex.join(["tensorboard", "--logdir", str(OUTPUT / "tensorboard"),
                  "--host", "0.0.0.0", "--port", "6006"]))
metrics_file = OUTPUT / "validation.jsonl"
if metrics_file.exists():
    points = [json.loads(line) for line in metrics_file.read_text().splitlines() if line.strip()]
    if points:
        table(["Step", "Group-averaged CE"], [(p["step"], f'{p["macro_ce"]:.4f}') for p in points])
        last = points[-1]
        table(["Latest validation: family", "Decisions", "Accuracy", "CE", "Score MAE"],
              [(f, int(v["n"]), f'{v["accuracy"]:.1%}', f'{v["ce"]:.4f}',
                "—" if v["ordinal_mae"] is None else f'{v["ordinal_mae"]:.4f}')
               for f, v in last["families"].items()])
else:
    print("No validation results are available for this run yet.")

## 7. Compare the base and adapted models side by side

Edit the examples below to try your own questions. **The correct answer is stored in `expected`** and displayed alongside predictions.

These are demonstrations, not benchmark evidence. The ordinal example expects `medium`, at position 1; the returned numerical score is an expectation and may be fractional.


In [ ]:
examples = [
    {"context": "Policy: escalate unresolved access failures. The account was locked yesterday, but access has been restored.",
     "instruction": "Should this access problem be escalated under the policy?",
     "question_type": "noul", "criteria": {"false": "Do not escalate.", "true": "Escalate."},
     "expected": "false"},
    {"context": "All approved invoices have been paid. Invoice A is approved.",
     "instruction": "Classify the hypothesis: Invoice A has been paid.", "question_type": "choice",
     "criteria": {"entailment": "Follows from the premise.", "neutral": "Cannot be determined.", "contradiction": "Contradicts the premise."},
     "expected": "entailment"},
    {"context": "Rubric: low = no affected users; medium = 1 to 9; high = 10 or more. Four users cannot log in.",
     "instruction": "Choose the severity according to the rubric.", "question_type": "score",
     "criteria": {"low": "No users.", "medium": "1 to 9 users.", "high": "10 or more users."},
     "expected": "medium"},
]

In [ ]:
RUN_COMPARISON = RUN_WORKFLOW
COMPARE_DEVICE = "cpu"  # "cuda" if your GPU is available.
if RUN_COMPARISON:
    from arbiter import ArbiterModel
    packages = [("Base", BASE), ("Adapted", OUTPUT / "best")]
    unavailable = [str(path) for _, path in packages if not (path / "model.safetensors").is_file()]
    if unavailable:
        print("Missing packages:", ", ".join(unavailable))
    else:
        comparison = []
        requests = [{k:v for k,v in e.items() if k != "expected"} for e in examples]
        for name, package in packages:
            model = ArbiterModel.from_pretrained(package, device=COMPARE_DEVICE)
            for i, (example, result) in enumerate(zip(examples, model.decide_batch(requests))):
                key = result["selected_key"]
                comparison.append([i+1, name, example["expected"], key,
                                   "✓" if key == example["expected"] else "Incorrect",
                                   f'{result["probabilities"][key]:.1%}'])
            del model
            gc.collect()
            if COMPARE_DEVICE.startswith("cuda"):
                import torch
                torch.cuda.empty_cache()
        table(["Example", "Model", "Expected", "Predicted", "Result", "Selected probability"], comparison)
else:
    print("Set RUN_COMPARISON to True once the adapted package is available.")

## 8. Evaluate the frozen candidate

Run this after selecting a candidate on validation. Use the same protocol and examples for both models. Evaluation includes the targeted skills **and** those to retain.

This cell prepares four evaluations: base/adapted × historical/public. Each invocation writes to a distinct directory. Existing prediction outputs are rejected; change `EVALUATION_NAME` for a new evaluation campaign.

Report gains and losses by family, multi-label results, and paired uncertainty estimates. Better validation CE does not guarantee better public F1. See the plan for statistical analysis and retention criteria.


In [ ]:
RUN_BENCHMARK = RUN_WORKFLOW
EVALUATION_NAME = RUN_NAME + "-evaluation-01"
if RUN_BENCHMARK and not (OUTPUT / "best/model.safetensors").is_file():
    print("No adapted package is available. Finish training first.")
else:
    for name, package in [("base", BASE), ("adapted", OUTPUT / "best")]:
        for suite, data in [("historical", HELDOUT[0]), ("public", HELDOUT[1])]:
            cmd = [sys.executable, "-m", "benchmark.run", "--model", "arbiter",
                   "--checkpoint", str(package), "--data", str(data), "--device", DEVICE,
                   "--output", str(ROOT / "results" / EVALUATION_NAME / name / suite)]
            if BENCHMARK_LIMIT:
                cmd += ["--limit", str(BENCHMARK_LIMIT)]
            launch(cmd, RUN_BENCHMARK)

## Troubleshooting

| Message or situation | What to do |
|---|---|
| Missing data | Prepare the JSONL files, update paths in Step 2, and rerun the audit |
| Overlap detected | Correct the training/validation split; preserve the benchmark |
| Input too long | Check the text and total token budget; there is no silent truncation |
| CUDA out of memory | Reduce `BATCH_SIZE`, adjust `ACCUMULATION`, and use a new run if parameters change |
| Nonempty output | Choose a new run name, or use `RESUME = True` for a compatible checkpoint |
| Manifest mismatch | Restore the original data, code, and parameters before resuming |
| Missing `best/` | Initial validation has not finished, or you are checking the wrong directory |
| Worse results | Keep the base; inspect validation families before revising the mixture |

**Pipeline validation status:** see the [end-to-end validation report](../docs/finetuning_validation.md) for the tested dataset, GPU run, resume and export checks. This is a technical workflow check, not evidence that fine-tuning improves benchmark scores.
